<a href="https://colab.research.google.com/github/1kaiser/Media-Segment-Depth-MLP/blob/main/STRING_3D_ViTTiny_Elayers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## string 3d ViT multilayer attention experiment

### folder creations

In [1]:
!pip install -q grain opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 5.0 MB/s eta 0:00:00


In [2]:
%cd /content/

/content


In [3]:
import os
import shutil
import glob


# Create the project structure
PROJECT_DIR = "vision_transformer_depth"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Project directories are set up successfully!")

✅ Project directories are set up successfully!


### model code

In [4]:
%cd /content/

/content


In [5]:
%%writefile vision_transformer_depth/string3d_experiment_pipeline.py

import os
import sys
import pickle
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob

import jax
import jax.numpy as jnp
from jax import random, jit, value_and_grad

import flax.linen as nn
import optax
import grain
from einops import rearrange
from tqdm import trange

# --- SECTION 2: DATA LOADING (GRAIN) ---
class DepthDataSource:
    def __init__(self, data_dir="."):
        self.pairs = [(f, os.path.join(data_dir, "depth_maps", os.path.basename(f))) for f in glob.glob(os.path.join(data_dir, "input_frames", "*.png")) if os.path.exists(os.path.join(data_dir, "depth_maps", os.path.basename(f)))]
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        inp_path, dep_path = self.pairs[idx]
        inp_rgb = cv2.cvtColor(cv2.imread(inp_path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        dep = cv2.imread(dep_path, cv2.IMREAD_GRAYSCALE)
        inp_rgb, dep = inp_rgb.astype(np.float32)/255.0, dep.astype(np.float32)/255.0
        h, w, _ = inp_rgb.shape
        y, x = max(0, (h - 224) // 2), max(0, (w - 224) // 2)
        inp_cropped, dep_cropped = inp_rgb[y:y+224, x:x+224], dep[y:y+224, x:x+224]
        return {'image': inp_cropped, 'depth': dep_cropped[..., None]}

def create_data_iterator(data_dir, batch_size, seed=42):
    source = DepthDataSource(data_dir=data_dir)
    if not source.pairs: return iter([])
    dataset = (grain.MapDataset.source(source).shuffle(seed=seed).batch(batch_size, drop_remainder=True).repeat().to_iter_dataset())
    return iter(dataset)

# --- SECTION 3: MODEL ARCHITECTURE (FLAX) ---
class StringPositionEmbedding3D(nn.Module):
    embed_dim: int
    @nn.compact
    def __call__(self, x, x_coords, y_coords, z_coords):
        S = self.param('S_cayley', nn.initializers.normal(0.01), (self.embed_dim, self.embed_dim))
        S_antisym, I = (S - S.T) / 2.0, jnp.eye(self.embed_dim, dtype=x.dtype)
        P = jnp.linalg.solve(I + S_antisym, I - S_antisym)
        x_transformed = jnp.matmul(x, P.T)
        freqs = 1.0 / (10000 ** (jnp.arange(0, self.embed_dim // 2, dtype=jnp.float32) * 2 / self.embed_dim))
        angles = jnp.einsum('i,j->ij', x_coords, freqs) + jnp.einsum('i,j->ij', y_coords, freqs) + jnp.einsum('i,j->ij', z_coords, freqs)
        cos_vals, sin_vals = jnp.repeat(jnp.cos(angles), 2, axis=-1), jnp.repeat(jnp.sin(angles), 2, axis=-1)
        x1, x2 = jnp.split(x_transformed, 2, axis=-1)
        return x_transformed * cos_vals[None, None, :, :] + jnp.concatenate([-x2, x1], axis=-1) * sin_vals[None, None, :, :]

class String3DViTAttention(nn.Module):
    num_heads: int; embed_dim: int; patch_size: int = 16
    @nn.compact
    def __call__(self, x, depth_map):
        b, s, e = x.shape; head_dim = self.embed_dim // self.num_heads
        string_encoder = StringPositionEmbedding3D(embed_dim=head_dim, name="string_3d_encoder")
        qkv = nn.Dense(features=self.embed_dim * 3, use_bias=False, name="qkv")(x)
        q, k, v = jnp.split(qkv, 3, axis=-1)
        q, k, v = [q.reshape(b, s, self.num_heads, head_dim).transpose(0, 2, 1, 3) for q in (q, k, v)]
        n_patches_per_dim = int(np.sqrt(s - 1))
        y_grid, x_grid = jnp.meshgrid(jnp.arange(n_patches_per_dim), jnp.arange(n_patches_per_dim), indexing='ij')
        x_coords, y_coords = jnp.concatenate([jnp.zeros(1), x_grid.flatten() + 1]), jnp.concatenate([jnp.zeros(1), y_grid.flatten() + 1])
        patch_depths = jnp.mean(rearrange(depth_map, 'b (h p1) (w p2) c -> b (h w) p1 p2 c', p1=self.patch_size, p2=self.patch_size), axis=(2,3)).squeeze(-1)
        z_coords_BL = jnp.concatenate([jnp.zeros((b, 1)), patch_depths], axis=1)
        z_coords_mean = jnp.mean(z_coords_BL, axis=0)
        q_final, k_final = string_encoder(q, x_coords, y_coords, z_coords_mean), string_encoder(k, x_coords, y_coords, z_coords_mean)
        scores = (q_final @ k_final.transpose(0, 1, 3, 2)) / jnp.sqrt(head_dim)
        attn = nn.softmax(scores, axis=-1)
        output = (attn @ v).transpose(0, 2, 1, 3).reshape(b, s, e)
        return nn.Dense(features=self.embed_dim, name="out_proj")(output)

class ViTTransformerBlock(nn.Module):
    embed_dim: int; num_heads: int
    @nn.compact
    def __call__(self, x, depth_map):
        attn_out = String3DViTAttention(embed_dim=self.embed_dim, num_heads=self.num_heads, name="attention")(nn.LayerNorm()(x), depth_map)
        x = x + attn_out
        mlp_out = nn.Dense(features=self.embed_dim * 4)(nn.LayerNorm()(x))
        mlp_out = nn.gelu(mlp_out)
        mlp_out = nn.Dense(features=self.embed_dim)(mlp_out)
        return x + mlp_out

class DepthHead(nn.Module):
    @nn.compact
    def __call__(self, patch_features):
        n_patches_per_dim = int(patch_features.shape[1] ** 0.5)
        x = rearrange(patch_features, 'b (h w) c -> b h w c', h=n_patches_per_dim)
        x = nn.ConvTranspose(features=128, kernel_size=(2, 2), strides=(2, 2))(x)
        x = nn.ConvTranspose(features=64, kernel_size=(2, 2), strides=(2, 2))(x)
        x = nn.ConvTranspose(features=32, kernel_size=(2, 2), strides=(2, 2))(x)
        x = nn.ConvTranspose(features=16, kernel_size=(2, 2), strides=(2, 2))(x)
        return nn.Conv(features=1, kernel_size=(1, 1))(x)

class ViTForDepth_String3D(nn.Module):
    # --- *** NEW: Parameters are now instance attributes *** ---
    patch_size: int; num_layers: int; embed_dim: int; num_heads: int
    @nn.compact
    def __call__(self, image, depth_map, train: bool = True):
        patches = nn.Conv(features=self.embed_dim, kernel_size=(self.patch_size, self.patch_size), strides=(self.patch_size, self.patch_size))(image)
        patches = patches.reshape(patches.shape[0], -1, self.embed_dim)
        cls_token = self.param('cls_token', nn.initializers.zeros, (1, 1, self.embed_dim))
        x = jnp.concatenate([jnp.tile(cls_token, (patches.shape[0], 1, 1)), patches], axis=1)
        x = nn.Dropout(rate=0.1, deterministic=not train)(x)
        for i in range(self.num_layers):
            x = ViTTransformerBlock(embed_dim=self.embed_dim, num_heads=self.num_heads, name=f"layer_{i}")(x, depth_map)
        return DepthHead(name="depth_head")(nn.LayerNorm()(x[:, 1:, :]))

# --- SECTION 4: TRAINING & CHECKPOINTING SCRIPT ---
def train_model(config):
    print(f"\n🚀 Starting Training: {config['num_layers']} Layers...")
    key, dropout_key = random.split(random.PRNGKey(config['seed']))
    data_iter = create_data_iterator(config['data_dir'], config['batch_size'], config['seed'])

    try: batch = next(data_iter)
    except StopIteration:
        print("Error: Data iterator is empty."); return None, []

    # --- *** NEW: Instantiate model with config values *** ---
    model = ViTForDepth_String3D(
        patch_size=16,
        num_layers=config['num_layers'],
        embed_dim=config['embed_dim'],
        num_heads=config['num_heads']
    )
    params = model.init(key, batch['image'], batch['depth'], train=False)['params']
    optimizer = optax.adam(config['lr'])
    opt_state = optimizer.init(params)

    @jit
    def train_step(params, opt_state, batch, dropout_rng):
        images, gt_depths = batch['image'], batch['depth']
        def loss_fn(p):
            pred_depths = model.apply({'params': p}, images, gt_depths, train=True, rngs={'dropout': dropout_rng})
            return jnp.mean(jnp.abs(pred_depths.squeeze() - gt_depths.squeeze()))
        loss, grads = value_and_grad(loss_fn)(params)
        updates, new_opt_state = optimizer.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), new_opt_state, loss

    all_losses = []
    pbar = trange(config['steps'], desc=f"🔥 Training ({config['num_layers']} Layers)")
    for step in pbar:
        dropout_key, step_key = random.split(dropout_key)
        batch = next(data_iter)
        params, opt_state, loss = train_step(params, opt_state, batch, step_key)
        loss_val = loss.item()
        all_losses.append(loss_val)
        pbar.set_postfix(loss=f"{loss_val:.4f}")

    with open(config['checkpoint_path'], "wb") as f: pickle.dump(params, f)
    print(f"✅ Training finished. Parameters saved to {config['checkpoint_path']}")
    return config['checkpoint_path'], all_losses

# --- SECTION 5: INFERENCE SCRIPT ---
def run_inference(config, sample_image_path):
    checkpoint_path = config['checkpoint_path']
    print(f"\n🧠 Running Inference for {config['num_layers']}-Layer Model...")
    with open(checkpoint_path, 'rb') as f: params = pickle.load(f)

    # --- *** NEW: Instantiate model with config values *** ---
    model = ViTForDepth_String3D(
        patch_size=16,
        num_layers=config['num_layers'],
        embed_dim=config['embed_dim'],
        num_heads=config['num_heads']
    )

    img = cv2.cvtColor(cv2.imread(sample_image_path), cv2.COLOR_BGR2RGB)
    img = (img.astype(np.float32) / 255.0)[16:240, 16:240]
    img_batch = jnp.expand_dims(img, 0)

    @jit
    def predict(p, img):
        dummy_depth = jnp.zeros_like(img[:, :, :, :1])
        return model.apply({'params': p}, img, dummy_depth, train=False)

    predicted_depth = predict(params, img_batch).squeeze()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    fig.suptitle(f"Depth Prediction using {config['num_layers']}-Layer ViT", fontsize=14)
    ax1.imshow(img); ax1.set_title("Input RGB"); ax1.axis('off')
    ax2.imshow(predicted_depth, cmap='magma'); ax2.set_title("Predicted Depth"); ax2.axis('off')

    output_name = os.path.join("outputs", f"prediction_{config['num_layers']}_layers.png")
    plt.savefig(output_name); plt.show()
    print(f"🖼️  Inference plot saved to: {output_name}")

# --- SECTION 6: MAIN EXECUTION BLOCK ---
if __name__ == "__main__":
    # --- *** NEW: Define the list of experiments to run *** ---
    layer_configs_to_test = [2, 4, 6, 12]
    all_results = {}

    for num_layers in layer_configs_to_test:
        config = {
            'seed': 42, 'data_dir': "./data", 'batch_size': 8,
            'steps': 5000, 'lr': 3e-4, # Fewer steps for quicker comparison
            'num_layers': num_layers,
            'embed_dim': 192,
            'num_heads': 3,
            'checkpoint_path': f"./checkpoints/string3d_model_{num_layers}_layers.pkl"
        }

        final_checkpoint, losses = train_model(config)
        all_results[f'{num_layers} Layers'] = losses

        if final_checkpoint:
            run_inference(config, sample_image_path="./data/input_frames/frame_0014.png")

    # --- *** NEW: Plot a final comparison of all loss curves *** ---
    plt.figure(figsize=(12, 7))
    for name, losses in all_results.items():
        plt.plot(losses, label=name)
    plt.title("Training Loss Comparison by Number of Layers")
    plt.xlabel("Training Step")
    plt.ylabel("L1 Loss")
    plt.legend()
    plt.grid(True)
    comparison_path = os.path.join("outputs", "loss_curve_comparison.png")
    plt.savefig(comparison_path)
    plt.show()
    print(f"📉 Final comparison plot saved to: {comparison_path}")

Writing vision_transformer_depth/string3d_experiment_pipeline.py


### frame dataset downloading

In [6]:
%cd /content/

/content


In [7]:
!wget -nc -q https://github.com/1kaiser/Media-Segment-Depth-MLP/releases/download/v0.2/input_depthMaps.zip
!unzip -o /content/input_depthMaps.zip > /dev/null 2>&1

In [8]:
import os
import shutil

# Move data into the project directory
if os.path.exists('/content/input_frames'):
    dest_path = os.path.join(DATA_DIR, 'input_frames')
    if os.path.exists(dest_path): shutil.rmtree(dest_path)
    shutil.move('/content/input_frames', dest_path)
if os.path.exists('/content/depth_maps'):
    dest_path = os.path.join(DATA_DIR, 'depth_maps')
    if os.path.exists(dest_path): shutil.rmtree(dest_path)
    shutil.move('/content/depth_maps', dest_path)

print("✅ Project data are set up successfully!")

✅ Project data are set up successfully!


### training cell

In [9]:
# Change to the project directory
%cd /content/vision_transformer_depth

# Run the full experiment pipeline
!python string3d_experiment_pipeline.py

/content/vision_transformer_depth

🚀 Starting Training: 2 Layers...
🔥 Training (2 Layers): 100% 5000/5000 [03:23<00:00, 24.58it/s, loss=0.0352]
✅ Training finished. Parameters saved to ./checkpoints/string3d_model_2_layers.pkl

🧠 Running Inference for 2-Layer Model...
Figure(1000x500)
🖼️  Inference plot saved to: outputs/prediction_2_layers.png

🚀 Starting Training: 4 Layers...
🔥 Training (4 Layers): 100% 5000/5000 [03:41<00:00, 22.56it/s, loss=0.0290]
✅ Training finished. Parameters saved to ./checkpoints/string3d_model_4_layers.pkl

🧠 Running Inference for 4-Layer Model...
Figure(1000x500)
🖼️  Inference plot saved to: outputs/prediction_4_layers.png

🚀 Starting Training: 6 Layers...
🔥 Training (6 Layers): 100% 5000/5000 [03:41<00:00, 22.54it/s, loss=0.0268]
✅ Training finished. Parameters saved to ./checkpoints/string3d_model_6_layers.pkl

🧠 Running Inference for 6-Layer Model...
Figure(1000x500)
🖼️  Inference plot saved to: outputs/prediction_6_layers.png

🚀 Starting Training: 12 La

### satellite datat input

In [10]:
############# 3. DATA ACQUISITION (SHELL COMMANDS) #############

# 1. Download all necessary files
print("⬇️ Downloading dataset files...")
!rm -f /content/vision_transformer_depth/DigitalElevationModel_01.tif /content/folder01_part_aa /content/folder01_part_ab # Remove potentially corrupted files
!wget -nc --no-check-certificate -P /content/vision_transformer_depth https://github.com/1kaiser/APPEEAR-LDDAC-DATA-DOWNLOAD/releases/download/1/DigitalElevationModel_01.tif
!wget -nc --no-check-certificate -P /content https://github.com/1kaiser/APPEEAR-LDDAC-DATA-DOWNLOAD/releases/download/1/folder01_part_aa
!wget -nc --no-check-certificate -P /content https://github.com/1kaiser/APPEEAR-LDDAC-DATA-DOWNLOAD/releases/download/1/folder01_part_ab


# 2. Extract and organize the image data
print("📦 Extracting image files...")
!cat /content/folder01_part_* > /content/folder01.zip
!rm -rf /content/folder01_part_*
!mkdir -p /content/imagesfolder
!unzip -o /content/folder01.zip -d /content/imagesfolder > /dev/null 2>&1
!rm -rf /content/folder01.zip
!mv /content/imagesfolder/files/* /content/imagesfolder/ 2>/dev/null || true

# 3. Resample the DEM to match the satellite image resolution using the GDAL Python library
print("🗺️ Resampling DEM to match image resolution...")
import glob
from osgeo import gdal
import os

try:
    # Robustly find a sample tif file
    sample_tifs = glob.glob('/content/imagesfolder/*.tif')
    if not sample_tifs:
        raise FileNotFoundError("No .tif files found in /content/imagesfolder/ to use as a size reference.")

    sample_image_path = sample_tifs[0]
    print(f"Using '{os.path.basename(sample_image_path)}' as size reference.")

    # --- THE FIX: Use the GDAL Python API directly ---
    # Open the reference image to get its dimensions
    ref_ds = gdal.Open(sample_image_path)
    if ref_ds is None:
        raise RuntimeError(f"Failed to open reference image: {sample_image_path}")

    width = ref_ds.RasterXSize
    height = ref_ds.RasterYSize
    # Close the dataset
    ref_ds = None

    print(f"Detected image size: {width}x{height}. Resampling DEM...")

    # Perform the warp operation using the Python function
    gdal.Warp('/content/DEMx.tif', '/content/vision_transformer_depth/DigitalElevationModel_01.tif',
              width=width, height=height, resampleAlg='bilinear')

except Exception as e:
    print(f"⚠️ An error occurred during resampling. Error: {e}")
    print("Falling back to copying the original DEM.")
    # Check if the original DEM exists before attempting to copy
    if os.path.exists('/content/vision_transformer_depth/DigitalElevationModel_01.tif'):
        !cp /content/vision_transformer_depth/DigitalElevationModel_01.tif /content/DEMx.tif
    else:
        print("Original DEM not found. Cannot perform fallback copy.")


# --- Final Verification Step ---
if os.path.exists('/content/DEMx.tif'):
    print("✅ Data acquisition and preparation complete. DEMx.tif is ready.")
else:
    raise RuntimeError("FATAL ERROR: /content/DEMx.tif was not created. Cannot proceed.")

⬇️ Downloading dataset files...
--2025-08-15 09:17:09--  https://github.com/1kaiser/APPEEAR-LDDAC-DATA-DOWNLOAD/releases/download/1/DigitalElevationModel_01.tif
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/570306733/b4739f92-70b3-43e8-97e0-fac3d7a1be36?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-08-15T10%3A15%3A59Z&rscd=attachment%3B+filename%3DDigitalElevationModel_01.tif&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-08-15T09%3A15%3A29Z&ske=2025-08-15T10%3A15%3A59Z&sks=b&skv=2018-11-09&sig=p%2FtCgmNLNbPlXchqEvD5EOG4leo0nSRGg2zCRqz6rMU%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc1NTI0OTc

/usr/local/lib/python3.11/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


✅ Data acquisition and preparation complete. DEMx.tif is ready.


In [11]:
import os
import shutil
import glob
import numpy as np
from osgeo import gdal
import cv2

print("--- Starting Satellite Data Preparation ---")

# Step 1: Clean data directory
print("🧹 Cleaning data directory...")
data_dir_path = '/content/vision_transformer_depth/data'
if os.path.exists(data_dir_path):
    # Use shell command for robust recursive deletion
    !rm -rf {data_dir_path}/*
    print(f"Cleared contents of {data_dir_path}")
else:
    print(f"Data directory not found, creating: {data_dir_path}")
os.makedirs(data_dir_path, exist_ok=True) # Ensure the data directory exists


def gdal_to_numpy(path: str, xoff=0, yoff=0, xsize=None, ysize=None) -> np.ndarray:
    """Safely opens a GDAL raster and converts a region to a NumPy array."""
    raster_ds = gdal.Open(path)
    if raster_ds is None:
        raise FileNotFoundError(f"GDAL failed to open or find the file at path: {path}")
    # Read the specified region, default to full raster if size is None
    return raster_ds.ReadAsArray(xoff, yoff, xsize, ysize).astype(np.float32)

# Normalization for the depth target
normalize_depth = lambda band: (band - band.min()) / (band.max() - band.min()) if band.max() > band.min() else np.zeros_like(band)

# Define paths and band numbers for RGB
satellite_image_dir = '/content/imagesfolder/'
dem_path = '/content/DEMx.tif'
# Use bands 3, 2, 1 for RGB (MODIS bands) ["b01", "b02", "b03", "b04", "b05", "b06", "b07"]
rgb_bands_numbers = ["b05", "b04", "b03"]


# Step 2: Determine dimensions
print("⬇️ Determining dimensions from satellite image bands and DEM...")

# Get dimensions from a sample RGB band and the DEM
try:
    # Find a sample file for the first RGB band
    sample_image_path_glob = glob.glob(os.path.join(satellite_image_dir, f'*{rgb_bands_numbers[0]}*.tif'))
    if not sample_image_path_glob:
         raise FileNotFoundError(f"No {rgb_bands_numbers[0]}.tif files found in {satellite_image_dir}")

    sample_image = gdal_to_numpy(sample_image_path_glob[0])
    depth_map = gdal_to_numpy(dem_path)

    img_h, img_w = sample_image.shape[:2]
    dep_h, dep_w = depth_map.shape[:2]

    # Use the minimum dimensions to avoid index errors
    min_height = min(img_h, dep_h)
    min_width = min(img_w, dep_w)

except Exception as e:
    print(f"Error determining dimensions: {e}")
    raise RuntimeError("FATAL ERROR: Could not determine image and DEM dimensions.")


image_size = 224 # Define the tile size

# Calculate the number of tiles in each dimension
num_rows = min_height // image_size
num_cols = min_width // image_size

print(f"Minimum dimensions: {min_width}x{min_height}")
print(f"Number of tiles: {num_cols} columns x {num_rows} rows = {num_cols * num_rows} tiles.")

# Create output directories for processed tiles
output_frames_dir = os.path.join(data_dir_path, 'input_frames')
output_depth_dir = os.path.join(data_dir_path, 'depth_maps')

os.makedirs(output_frames_dir, exist_ok=True)
os.makedirs(output_depth_dir, exist_ok=True)

# Step 3 & 4: Tile, process, and save data
print("Processing and saving tiles...")
tile_count = 0
if num_rows > 0 and num_cols > 0:
    for row_idx in range(num_rows):
        for col_idx in range(num_cols):
            row_start = row_idx * image_size
            col_start = col_idx * image_size

            # Extract and normalize depth tile
            try:
                depth_tile_array = gdal_to_numpy(dem_path, xoff=col_start, yoff=row_start, xsize=image_size, ysize=image_size)
                depth_tile_normalized = normalize_depth(depth_tile_array)
            except Exception as e:
                print(f"Error processing depth tile at ({row_idx}, {col_idx}): {e}. Skipping tile.")
                continue # Skip this tile if depth processing fails


            # Extract image tiles for each RGB band and stack them
            image_bands_data = []
            bands_available_for_rgb = True
            try:
                for band_num_str in rgb_bands_numbers:
                     band_files = glob.glob(os.path.join(satellite_image_dir, f'*{band_num_str}*.tif'))
                     if band_files:
                         band_data = gdal_to_numpy(band_files[0], xoff=col_start, yoff=row_start, xsize=image_size, ysize=image_size)
                         if band_data.shape != (image_size, image_size):
                             print(f"Warning: Band {band_num_str} tile has incorrect shape {band_data.shape} at ({row_idx}, {col_idx}). Expected ({image_size}, {image_size}). Skipping tile.")
                             bands_available_for_rgb = False
                             break
                         image_bands_data.append(band_data)
                     else:
                         print(f"Error: Could not find file for band {band_num_str} for tiling at ({row_idx}, {col_idx}). Skipping tile.")
                         bands_available_for_rgb = False
                         break # Stop if any required RGB band is missing
            except Exception as e:
                 print(f"Error processing image bands tile at ({row_idx}, {col_idx}): {e}. Skipping tile.")
                 bands_available_for_rgb = False # Ensure we skip the tile


            # Stack the bands to create the RGB image tile if all required bands were found and shaped correctly
            if bands_available_for_rgb:
                image_tile_array = np.stack(image_bands_data, axis=-1)

                # Save the processed tiles
                frame_filename = os.path.join(output_frames_dir, f'frame_{tile_count:04d}.png')
                depth_filename = os.path.join(output_depth_dir, f'frame_{tile_count:04d}.png')

                # Convert numpy arrays to uint8 for saving as images
                # Scale image bands to 0-255 for uint8 saving.
                image_tile_scaled = (image_tile_array - image_tile_array.min()) / (image_tile_array.max() - image_tile_array.min()) * 255 if image_tile_array.max() > image_tile_array.min() else np.zeros_like(image_tile_array)
                image_tile_uint8 = image_tile_scaled.astype(np.uint8)

                # Scale depth tile to 0-255
                depth_tile_uint8 = (depth_tile_normalized * 255).astype(np.uint8)

                # Ensure the image tile has 3 channels for RGB
                if image_tile_uint8.shape[-1] != 3:
                    print(f"Warning: Final image tile for saving does not have 3 channels at ({row_idx}, {col_idx}). Shape: {image_tile_uint8.shape[-1]}. Skipping save.")
                    continue # Skip saving if not 3 channels

                cv2.imwrite(frame_filename, cv2.cvtColor(image_tile_uint8, cv2.COLOR_RGB2BGR)) # Save as BGR for OpenCV
                cv2.imwrite(depth_filename, depth_tile_uint8)

                tile_count += 1
            else:
                print(f"Skipping tile saving at ({row_idx}, {col_idx}) due to processing errors.")

else:
    print("Warning: No tiles to process (minimum dimension is less than image_size).")

print(f"--- Finished processing {tile_count} tiles ---")

# Step 5: Verify data structure
print("\n--- Verifying Data Structure ---")
# Check if directories exist
print(f"Checking directory: {output_frames_dir}")
if os.path.exists(output_frames_dir):
    print(f"Directory exists: {output_frames_dir}")
    # List contents of input_frames
    print(f"\nContents of {output_frames_dir}:")
    frames_list = os.listdir(output_frames_dir)
    print(frames_list[:10]) # Print only the first 10 files to avoid flooding the output
    if len(frames_list) > 10: print(f"... and {len(frames_list) - 10} more files.")
else:
    print(f"Directory does not exist: {output_frames_dir}")

print(f"\nChecking directory: {output_depth_dir}")
if os.path.exists(output_depth_dir):
    print(f"Directory exists: {output_depth_dir}")
    # List contents of depth_maps
    print(f"\nContents of {output_depth_dir}:")
    depth_list = os.listdir(output_depth_dir)
    print(depth_list[:10]) # Print only the first 10 files
    if len(depth_list) > 10: print(f"... and {len(depth_list) - 10} more files.")
else:
    print(f"Directory does not exist: {output_depth_dir}")

# Count the number of files in each directory
num_image_files = len([name for name in os.listdir(output_frames_dir) if os.path.isfile(os.path.join(output_frames_dir, name))]) if os.path.exists(output_frames_dir) else 0
num_depth_files = len([name for name in os.listdir(output_depth_dir) if os.path.isfile(os.path.join(output_depth_dir, name))]) if os.path.exists(output_depth_dir) else 0


print(f"\nNumber of image files found: {num_image_files}")
print(f"Number of depth files found: {num_depth_files}")

# Verify that the number of files match
if num_image_files == num_depth_files and num_image_files > 0:
    print("✅ Verification successful: Number of image files matches the number of depth files.")
else:
    print("❌ Verification failed: The number of image and depth files do not match or no files were found.")

print("--- Satellite Data Preparation Complete ---")

--- Starting Satellite Data Preparation ---
🧹 Cleaning data directory...
Cleared contents of /content/vision_transformer_depth/data
⬇️ Determining dimensions from satellite image bands and DEM...
Minimum dimensions: 15171x237
Number of tiles: 67 columns x 1 rows = 67 tiles.
Processing and saving tiles...
--- Finished processing 67 tiles ---

--- Verifying Data Structure ---
Checking directory: /content/vision_transformer_depth/data/input_frames
Directory exists: /content/vision_transformer_depth/data/input_frames

Contents of /content/vision_transformer_depth/data/input_frames:
['frame_0018.png', 'frame_0054.png', 'frame_0009.png', 'frame_0025.png', 'frame_0034.png', 'frame_0064.png', 'frame_0008.png', 'frame_0060.png', 'frame_0043.png', 'frame_0026.png']
... and 57 more files.

Checking directory: /content/vision_transformer_depth/data/depth_maps
Directory exists: /content/vision_transformer_depth/data/depth_maps

Contents of /content/vision_transformer_depth/data/depth_maps:
['frame_

### training cell

In [12]:
# Change to the project directory
%cd /content/vision_transformer_depth

# Run the full experiment pipeline
!python string3d_experiment_pipeline.py

/content/vision_transformer_depth

🚀 Starting Training: 2 Layers...
🔥 Training (2 Layers): 100% 5000/5000 [02:37<00:00, 31.79it/s, loss=0.0679]
✅ Training finished. Parameters saved to ./checkpoints/string3d_model_2_layers.pkl

🧠 Running Inference for 2-Layer Model...
Figure(1000x500)
🖼️  Inference plot saved to: outputs/prediction_2_layers.png

🚀 Starting Training: 4 Layers...
🔥 Training (4 Layers): 100% 5000/5000 [02:52<00:00, 29.04it/s, loss=0.0550]
✅ Training finished. Parameters saved to ./checkpoints/string3d_model_4_layers.pkl

🧠 Running Inference for 4-Layer Model...
Figure(1000x500)
🖼️  Inference plot saved to: outputs/prediction_4_layers.png

🚀 Starting Training: 6 Layers...
🔥 Training (6 Layers): 100% 5000/5000 [03:04<00:00, 27.10it/s, loss=0.0532]
✅ Training finished. Parameters saved to ./checkpoints/string3d_model_6_layers.pkl

🧠 Running Inference for 6-Layer Model...
Figure(1000x500)
🖼️  Inference plot saved to: outputs/prediction_6_layers.png

🚀 Starting Training: 12 La